# Sparse, dense and keyword retrieval strategies in Vespa

This notebook demonstrates the strategies defined in [`src/strategies.py`](src/strategies.py) that use **BGE-M3 sparse vectors**, alone and in combination with dense vectors and BM25 keyword search. Every strategy is shown on a query from [`vespa_sparse_test_dataset.json`](src/vespa_sparse_test_dataset.json) that was written to **favour it**: the YQL that was sent, the query's sparse tokens, the ranked hits with their match-features, and an explanation of why retrieval and ranking behave the way they do.

Each strategy has two parts:

* **retrieval** (the `where` clause) decides *which documents are candidates*. Operators joined with `or` return the union of their candidates.
* **ranking** (the rank profile in [`doc.sd`](simple_app/schemas/doc.sd)) decides *the order*, using exact scores from tensors / BM25.

| operator | finds | scores in rank profile |
|---|---|---|
| `userQuery()` | docs containing query words, whole-word (stemmed) match | `bm25(field)` |
| `nearestNeighbor(embedding, q)` | the `targetHits` closest dense vectors | `closeness` / dot product |
| `wand(sparse_tokens, {...})` | docs sharing **any** query token, pruned to top-k by weighted overlap | `sum(query(qs) * attribute(sparse_embedding))` |
| `dotProduct(sparse_tokens, {...})` | all docs sharing any query token | `rawScore(sparse_tokens)` |

Set `VESPA_URL` / `VESPA_PORT` if Vespa is not on `localhost:8080`. The expectations written in the markdown cells are the *design intent* of each example; the tables are produced live, so always check the numbers.

In [1]:
import os, sys, json
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from vespa.application import Vespa
from FlagEmbedding import BGEM3FlagModel

VESPA_URL = os.environ.get("VESPA_URL", "http://localhost")
VESPA_PORT = int(os.environ.get("VESPA_PORT", "8080"))
app = Vespa(url=VESPA_URL, port=VESPA_PORT)
model = BGEM3FlagModel("BAAI/bge-m3")
tokenizer = model.tokenizer

import pandas as pd
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

from constants import SPARSE_DATA_FILE
from data import load_dataset
from embeddings import BGEM3Embedder
from evaluation import QueryExecutor, BenchmarkRunner
from indexing import feed_documents
from strategies import STRATEGIES

dataset = load_dataset(SPARSE_DATA_FILE)
docs = {d["id"]: d for d in dataset["documents"]}
queries = {q["id"]: q for q in dataset["queries"]}
strategies = {s.id: s for s in STRATEGIES}

embedder = BGEM3Embedder()   # loads the model a second time; fine for a demo
executor = QueryExecutor(app)
print(len(docs), "documents,", len(queries), "queries")

41 documents, 20 queries


## Index the documents

Each document is fed with dense vectors, the sparse tensors and the integer weightedset (see [`indexing.py`](src/indexing.py)).

In [2]:
feed_documents(app, [dict(d) for d in dataset["documents"]], embedder=embedder)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


## Documents that make up the examples

Short groups of near-duplicates are deliberate. Dense vectors map "Release notes 8.314.12" and "Release notes 8.327.8" almost to the same point, so a version number has to be matched lexically; the model-type documents differ in one rare word.

In [3]:
pd.DataFrame([(d["id"], d["title"], d["category"]) for d in dataset["documents"]], columns=["id", "title", "category"])

,id,title,category
0,sp-001,Error VESP-4021 document exceeds maximum size,Errors
1,sp-002,Error VESP-4022 feed operation timed out,Errors
2,sp-003,Error VESP-4023 schema field type mismatch,Errors
3,sp-004,Error VESP-4120 content node unavailable,Errors
4,sp-005,Troubleshooting failed feed operations,Errors
5,sp-006,Re-ranking candidates in the second phase,Ranking
6,sp-007,Cross-encoder scoring for top candidates,Ranking
7,sp-008,Autoscaling content clusters,Operations
8,sp-009,Manual capacity planning for clusters,Operations
9,sp-010,Nearest neighbor search with HNSW graphs,Vectors


## Helper that shows a strategy on a query

In [4]:
def show(strategy_id, query_id, hits=5):
    strategy, query = strategies[strategy_id], queries[query_id]
    response = executor.search(strategy, query["text"], embedder)
    relevant = query["relevance_scores"]

    print(f"Strategy : {strategy.id}  (rank profile: {strategy.rank_profile})")
    print(f"Query    : {query['text']!r}   [designed to favour: {query['favours']}]")
    print(f"Why      : {query['why']}")
    yql = executor.last_yql
    print("YQL      :", yql if len(yql) < 350 else yql[:350] + " ...")
    sparse = executor.last_sparse_query
    if sparse:
        tokens = sorted(((tokenizer.decode([int(t)]).strip(), round(w, 3)) for t, w in sparse.items()), key=lambda x: -x[1])
        print("Sparse query tokens (token, weight):", tokens)

    rows = []
    for rank, hit in enumerate(response.hits[:hits], start=1):
        doc_id = hit["fields"]["id"]
        features = {k: round(v, 3) for k, v in hit["fields"].get("matchfeatures", {}).items()}
        rows.append((rank, doc_id, "yes (%s)" % relevant[doc_id] if doc_id in relevant else "", round(hit["relevance"], 3),
                     docs[doc_id]["title"][:55], features))
    missing = [d for d in relevant if d not in {h["fields"]["id"] for h in response.hits}]
    if missing:
        print("Relevant documents NOT retrieved:", missing)
    return pd.DataFrame(rows, columns=["rank", "id", "relevant", "relevance", "title", "match-features"])

def compare(query_id, strategy_ids, k=3):
    """Top-k ids of several strategies side by side; relevant ids are marked with *."""
    query = queries[query_id]
    relevant = set(query["relevant_ids"])
    out = {}
    for sid in strategy_ids:
        ids = executor.run_query(strategies[sid], query["text"], embedder)[:k]
        out[sid] = " ".join(("*" if i in relevant else "") + i for i in ids) or "(no hits)"
    return pd.Series(out, name=f"{query['text']!r}")

## Baselines for reference

Before the sparse strategies, the existing keyword and dense strategies show *what they are up against*. Whole-word BM25 cannot match sub-words or spelling variants, and dense vectors cannot tell near-identical documents apart.

In [5]:
compare("s17", ["bm25_title_body", "dense_embedding", "sparse_wand"])

bm25_title_body    *sp-035 sp-012 sp-037
dense_embedding    *sp-035 sp-033 sp-034
sparse_wand        *sp-035 sp-029 sp-031
Name: 'what shipped in 8.352.3', dtype: object

In [6]:
compare("s18", ["bm25_title_body", "dense_embedding", "sparse_wand"])

bm25_title_body                  *sp-030
dense_embedding    sp-031 *sp-030 sp-034
sparse_wand        *sp-030 sp-034 sp-029
Name: 'VSP-90427', dtype: object

In [7]:
compare("s22", ["bm25_title_body", "dense_embedding", "sparse_wand"])

No hits returned for query 'hnswgraph' (strategy=bm25_title_body).


bm25_title_body                (no hits)
dense_embedding    *sp-010 sp-019 sp-026
sparse_wand                      *sp-010
Name: 'hnswgraph', dtype: object

* `s17` (*"what shipped in 8.352.3"*): the version number identifies one release note; all three signals can find it, filler words only add noise.
* `s18` (*"VSP-90427"*): the eight release notes have nearly identical dense vectors, so dense search may rank another note first. The identifier is a lexical fact that keyword and sparse retrieval get right.
* `s22` (*"hnswgraph"*): no whole word matches "HNSW graphs", so BM25 finds nothing. Sparse vectors share sub-word tokens and dense vectors capture the meaning, so both succeed.

# 1. Sparse only

## `sparse_wand`

**Retrieval:** `wand(sparse_tokens, {...})` over every query token with `targetHits` 10. WAND walks the posting lists of the weighted tokens and only fully evaluates documents whose upper-bound score can still beat the current top-k, so it is much cheaper than scoring all documents that share a token.

**Ranking:** `sparse_dot` = `sum(query(qs) * attribute(sparse_embedding))`, the exact float dot product. The weightedset used for retrieval is integer-quantised; ranking recomputes with the real weights.

**Favoured by:** identifiers and rare words. `VSP-90427` is split in sub-word tokens; the document that contains all of those pieces gets the largest sum of products, while the other release notes only share the common words.

In [8]:
show("sparse_wand", "s18")

Strategy : sparse_wand  (rank profile: sparse_dot)
Query    : 'VSP-90427'   [designed to favour: keyword]
Why      : Ticket id: dense vectors of the release notes are almost identical.
YQL      : select * from doc where ([{"targetHits": 10}]wand(sparse_tokens, {"81": 165, "7008": 226, "73238": 209, "617": 180, "3768": 285}))
Sparse query tokens (token, weight): [('27', 0.285), ('sp', 0.226), ('-90', 0.209), ('4', 0.18), ('v', 0.165)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-030,yes (2.0),0.153,Release notes 8.314.17,{'sparse_score': 0.153}
1,2,sp-034,,0.121,Release notes 8.340.9,{'sparse_score': 0.121}
2,3,sp-029,,0.092,Release notes 8.314.12,{'sparse_score': 0.092}
3,4,sp-031,,0.079,Release notes 8.320.4,{'sparse_score': 0.079}
4,5,sp-032,,0.076,Release notes 8.327.8,{'sparse_score': 0.076}


## `sparse_wand_pruned`

**Retrieval:** the same `wand`, but only the **6 heaviest query tokens** are used, both in the YQL and in the ranking tensor.

**Ranking:** `sparse_dot` on the pruned tensor.

**Trade-off:** fewer terms means fewer posting lists to traverse. For a question such as *"what shipped in 8.352.3"* the version tokens carry the highest weights, filler words are discarded and precision typically improves; for a query where many medium-weight words all matter, pruning loses recall.

In [9]:
show("sparse_wand_pruned", "s17")

Strategy : sparse_wand_pruned  (rank profile: sparse_dot)
Query    : 'what shipped in 8.352.3'   [designed to favour: keyword]
Why      : Version number inside a question.
YQL      : select * from doc where ([{"targetHits": 10}]wand(sparse_tokens, {"120883": 266, "5843": 257, "3217": 204, "6544": 202, "48398": 149, "2367": 112}))
Sparse query tokens (token, weight): [('2.3', 0.266), ('35', 0.257), ('8.', 0.204), ('shi', 0.202), ('pped', 0.149), ('what', 0.112)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-035,yes (2.0),0.179,Release notes 8.352.3,{'sparse_score': 0.179}
1,2,sp-029,,0.045,Release notes 8.314.12,{'sparse_score': 0.045}
2,3,sp-031,,0.044,Release notes 8.320.4,{'sparse_score': 0.044}
3,4,sp-030,,0.042,Release notes 8.314.17,{'sparse_score': 0.042}
4,5,sp-032,,0.041,Release notes 8.327.8,{'sparse_score': 0.041}


## `sparse_dotproduct`

**Retrieval:** the `dotProduct` operator - any document sharing at least one token is a candidate, without top-k pruning.

**Ranking:** `sparse_rawscore` = `rawScore(sparse_tokens)`, the integer dot product computed by the operator itself, so no tensor multiplication is needed in the rank profile.

**Favoured by:** spelling variants. `hnswgraph` has no whole-word match with `HNSW graphs`, so BM25 returns nothing, but the learned sparse vectors share sub-word tokens. Results should match `sparse_wand`, which is the point: `wand` is an optimisation of the same score.

In [10]:
show("sparse_dotproduct", "s22")

Strategy : sparse_dotproduct  (rank profile: sparse_rawscore)
Query    : 'hnswgraph'   [designed to favour: sparse]
Why      : Merged spelling of 'HNSW graphs': BM25 has no whole-word match; sub-word tokens overlap.
YQL      : select * from doc where dotProduct(sparse_tokens, {"1096": 175, "1779": 166, "434": 202, "41382": 302})
Sparse query tokens (token, weight): [('graph', 0.302), ('w', 0.202), ('h', 0.175), ('ns', 0.166)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-010,yes (2.0),137861.0,Nearest neighbor search with HNSW graphs,{'sparse_raw': 137861.0}


## `sparse_field_weighted`

**Retrieval:** `wand` on the full-text sparse tokens.

**Ranking:** `sparse_field_weighted` = `2 * dot(title_sparse) + dot(sparse_embedding)`. The query tensor is multiplied with two document tensors; the title match counts double - analogous to the boosted BM25 profiles.

**Favoured by:** a rare word that occurs in the *title* of the right document (`CatBoost`): its title weight is added twice. Since all models documents have the same body wording, the title is the only discriminating field.

In [11]:
show("sparse_field_weighted", "s19")

Strategy : sparse_field_weighted  (rank profile: sparse_field_weighted)
Query    : 'CatBoost second phase'   [designed to favour: keyword]
Why      : Rare model name among near-duplicate documents: a lexical match decides, dense similarity may blur them.
YQL      : select * from doc where ([{"targetHits": 10}]wand(sparse_tokens, {"7515": 179, "837": 157, "5510": 165, "17932": 163, "93402": 186}))
Sparse query tokens (token, weight): [('phase', 0.186), ('cat', 0.179), ('ost', 0.165), ('second', 0.163), ('bo', 0.157)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-037,yes (2.0),0.420,Running CatBoost models in the second phase,"{'sparse_score': 0.145, 'title_sparse_score': 0.137}"
1,2,sp-013,,0.296,Running XGBoost models in the second phase,"{'sparse_score': 0.099, 'title_sparse_score': 0.098}"
2,3,sp-040,,0.211,Running linear regression models in the second phase,"{'sparse_score': 0.068, 'title_sparse_score': 0.071}"
3,4,sp-006,,0.210,Re-ranking candidates in the second phase,"{'sparse_score': 0.07, 'title_sparse_score': 0.07}"
4,5,sp-041,,0.207,Running Keras models in the second phase,"{'sparse_score': 0.069, 'title_sparse_score': 0.069}"


# 2. Sparse + dense

## `sparse_retrieve_dense_rank`

**Retrieval:** sparse `wand` only - candidates must share a token with the query.

**Ranking:** `sparse_dense` = `dense dot product + 0.3 * sparse dot product` on those candidates. Dense similarity is computed from the stored vector (it is *not* an ANN search).

**Weak spot:** the query *"make results come back sooner"* shares few words with the relevant document ("Reducing query latency"). Only candidates that WAND found can be ranked, so a paraphrase without any shared token would be lost. Here `results`, `back`, `come` are shared with enough documents that the dense score can pick the right one from the candidates; compare with `sparse_dense` below, which also retrieves with ANN.

In [12]:
show("sparse_retrieve_dense_rank", "s9")

Strategy : sparse_retrieve_dense_rank  (rank profile: sparse_dense)
Query    : 'make results come back sooner'   [designed to favour: dense]
Why      : No shared vocabulary with the relevant document; only semantics link them.
YQL      : select * from doc where ([{"targetHits": 10}]wand(sparse_tokens, {"3249": 216, "50339": 296, "1380": 143, "4420": 223, "33662": 208, "56": 175}))
Sparse query tokens (token, weight): [('results', 0.296), ('back', 0.223), ('make', 0.216), ('soon', 0.208), ('er', 0.175), ('come', 0.143)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-016,yes (2.0),0.661,Reducing query latency,"{'dense_score': 0.621, 'sparse_score': 0.131}"
1,2,sp-017,,0.549,Designing a search results page,"{'dense_score': 0.524, 'sparse_score': 0.083}"
2,3,sp-027,,0.508,Hybrid ranking with several signals,"{'dense_score': 0.504, 'sparse_score': 0.012}"
3,4,sp-002,,0.464,Error VESP-4022 feed operation timed out,"{'dense_score': 0.462, 'sparse_score': 0.007}"
4,5,sp-019,,0.451,Handling users who misspell words,"{'dense_score': 0.447, 'sparse_score': 0.015}"


## `dense_retrieve_sparse_rank`

**Retrieval:** dense `nearestNeighbor` only (`targetHits` 10).

**Ranking:** `dense_retrieve_sparse_rank` ranks the ANN candidates by sparse dot product, exposing the dense closeness as a match-feature.

**Idea:** use the dense index for recall of paraphrases and the sparse vector as a term-sensitive scorer. The query *"new content searchable right away"* is a paraphrase, so the candidate set comes from the semantics.

In [13]:
show("dense_retrieve_sparse_rank", "s11")

Strategy : dense_retrieve_sparse_rank  (rank profile: dense_retrieve_sparse_rank)
Query    : 'new content searchable right away'   [designed to favour: dense]
Why      : Paraphrase of 'fresh as content changes'.
YQL      : select * from doc where ([{"targetHits": 10}]nearestNeighbor(embedding, q))
Sparse query tokens (token, weight): [('content', 0.278), ('search', 0.26), ('away', 0.235), ('new', 0.205), ('able', 0.195), ('right', 0.146)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-018,yes (2.0),0.133,Keeping answers fresh as content changes,"{'closeness(field,embedding)': 0.532, 'sparse_score': 0.133}"
1,2,sp-032,,0.126,Release notes 8.327.8,"{'closeness(field,embedding)': 0.491, 'sparse_score': 0.126}"
2,3,sp-034,,0.125,Release notes 8.340.9,"{'closeness(field,embedding)': 0.489, 'sparse_score': 0.125}"
3,4,sp-030,,0.123,Release notes 8.314.17,"{'closeness(field,embedding)': 0.491, 'sparse_score': 0.123}"
4,5,sp-035,,0.122,Release notes 8.352.3,"{'closeness(field,embedding)': 0.488, 'sparse_score': 0.122}"


## `sparse_dense`

**Retrieval:** `wand(...) or nearestNeighbor(embedding, q)` - the *union* of the sparse and dense candidates.

**Ranking:** `sparse_dense` = `dense + 0.3 * sparse` (the weights follow the dense/sparse mix recommended for BGE-M3).

**Favoured by:** the German query *"partielle Updates ohne Neuindexierung"*. The multilingual dense vector links it to the English documents about partial updates; the sparse part adds credit for the German document that shares sub-words.

In [14]:
show("sparse_dense", "s12")

Strategy : sparse_dense  (rank profile: sparse_dense)
Query    : 'partielle Updates ohne Neuindexierung'   [designed to favour: dense]
Why      : German query; relevant documents share few tokens, a multilingual embedding links them.
YQL      : select * from doc where ([{"targetHits": 10}]wand(sparse_tokens, {"2878": 164, "2118": 113, "114419": 256, "10938": 127, "3717": 104, "4711": 215, "13976": 84})) or ([{"targetHits": 10}]nearestNeighbor(embedding, q))
Sparse query tokens (token, weight): [('updates', 0.256), ('index', 0.215), ('parti', 0.164), ('ohne', 0.127), ('elle', 0.113), ('neu', 0.104), ('ierung', 0.084)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-025,yes (2.0),0.719,Vespa Aktualisierung von Dokumenten,"{'dense_score': 0.681, 'sparse_score': 0.126}"
1,2,sp-018,yes (1.0),0.711,Keeping answers fresh as content changes,"{'dense_score': 0.677, 'sparse_score': 0.111}"
2,3,sp-032,,0.522,Release notes 8.327.8,"{'dense_score': 0.522, 'sparse_score': 0.0}"
3,4,sp-031,,0.501,Release notes 8.320.4,"{'dense_score': 0.501, 'sparse_score': 0.0}"
4,5,sp-035,,0.498,Release notes 8.352.3,"{'dense_score': 0.498, 'sparse_score': 0.0}"


# 3. Sparse + keyword

## `sparse_keyword`

**Retrieval:** `userQuery() or wand(...)`. The keyword branch matches whole (stemmed) words, the sparse branch adds sub-word matches.

**Ranking:** `sparse_keyword` = `sparse dot + 0.1 * (bm25(title) + bm25(body))`. BM25 scores are an order of magnitude larger than sparse scores, hence the small factor.

**Favoured by:** *"what does VESP-4023 mean"*: the identifier is the decisive evidence and both signals agree, which stabilises the order. The match-features show the BM25 values.

In [15]:
show("sparse_keyword", "s3")

Strategy : sparse_keyword  (rank profile: sparse_keyword)
Query    : 'what does VESP-4023 mean'   [designed to favour: keyword]
Why      : Identifier inside a question; sparse and BM25 key on the code.
YQL      : select * from doc where userQuery() or ([{"targetHits": 10}]wand(sparse_tokens, {"2367": 90, "14602": 75, "173": 217, "7008": 248, "41734": 208, "3742": 306, "29459": 141}))
Sparse query tokens (token, weight): [('23', 0.306), ('sp', 0.248), ('ve', 0.217), ('-40', 0.208), ('mean', 0.141), ('what', 0.09), ('does', 0.075)]
Relevant documents NOT retrieved: ['sp-005']


,rank,id,relevant,relevance,title,match-features
0,1,sp-003,yes (2.0),1.054,Error VESP-4023 schema field type mismatch,"{'bm25_body': 5.588, 'bm25_title': 2.931, 'sparse_score': 0.202}"
1,2,sp-002,,0.395,Error VESP-4022 feed operation timed out,"{'bm25_body': 2.681, 'bm25_title': 0.0, 'sparse_score': 0.127}"
2,3,sp-001,,0.118,Error VESP-4021 document exceeds maximum size,"{'bm25_body': 0.0, 'bm25_title': 0.0, 'sparse_score': 0.118}"
3,4,sp-004,,0.093,Error VESP-4120 content node unavailable,"{'bm25_body': 0.0, 'bm25_title': 0.0, 'sparse_score': 0.093}"
4,5,sp-031,,0.054,Release notes 8.320.4,"{'bm25_body': 0.0, 'bm25_title': 0.0, 'sparse_score': 0.054}"


# 4. Sparse + dense + keyword

## `sparse_dense_keyword`

**Retrieval:** `userQuery() or wand(...) or nearestNeighbor(...)` - three candidate generators.

**Ranking:** `sparse_dense_keyword` = `dense + 0.3 * sparse + 0.1 * (bm25(title) + bm25(body))`.

**Favoured by:** *"fix VESP-4022 by making feeding quicker"*. One relevant document is found through the identifier (lexical/sparse), another through the paraphrase "feeding quicker" (dense). Dense alone tends to rank the identifier document lower, because error documents all look alike to it; the keyword and sparse signals lift it, while the dense branch contributes the general troubleshooting document.

In [16]:
show("sparse_dense_keyword", "s13")

Strategy : sparse_dense_keyword  (rank profile: sparse_dense_keyword)
Query    : 'fix VESP-4022 by making feeding quicker'   [designed to favour: hybrid]
Why      : Needs the identifier (lexical) and the paraphrase 'feeding quicker' (semantic).
YQL      : select * from doc where userQuery() or ([{"targetHits": 10}]wand(sparse_tokens, {"30022": 222, "173": 191, "7008": 181, "41734": 169, "4015": 216, "390": 103, "20662": 187, "61590": 235, "214": 143, "63773": 213, "56": 180})) or ([{"targetHits": 10}]nearestNeighbor(embedding, q))
Sparse query tokens (token, weight): [('feed', 0.235), ('fix', 0.222), ('22', 0.216), ('quick', 0.213), ('ve', 0.191), ('making', 0.187), ('sp', 0.181), ('er', 0.18), ('-40', 0.169), ('ing', 0.143), ('by', 0.103)]
Relevant documents NOT retrieved: ['sp-016']


,rank,id,relevant,relevance,title,match-features
0,1,sp-002,yes (2.0),1.550,Error VESP-4022 feed operation timed out,"{'bm25_body': 4.729, 'bm25_title': 4.13, 'dense_score': 0.602, 'sparse_score..."
1,2,sp-005,yes (1.0),0.705,Troubleshooting failed feed operations,"{'bm25_body': 1.231, 'bm25_title': 1.488, 'dense_score': 0.416, 'sparse_scor..."
2,3,sp-001,,0.689,Error VESP-4021 document exceeds maximum size,"{'bm25_body': 1.209, 'bm25_title': 0.0, 'dense_score': 0.524, 'sparse_score'..."
3,4,sp-010,,0.657,Nearest neighbor search with HNSW graphs,"{'bm25_body': 2.95, 'bm25_title': 0.0, 'dense_score': 0.362, 'sparse_score':..."
4,5,sp-031,,0.648,Release notes 8.320.4,"{'bm25_body': 1.575, 'bm25_title': 0.0, 'dense_score': 0.479, 'sparse_score'..."


## `sparse_dense_keyword_rerank`

**Retrieval:** the same three-way union, `targetHits` 20.

**First phase:** `dense + 0.3 * sparse` - cheap, computed for every candidate.

**Second phase** (top 100 by first-phase score): adds `0.1 * (2*bm25(title) + bm25(body))` and `0.5 * fieldMatch(title)`. `fieldMatch` is expensive, so it only runs on the shortlist. Use this pattern when a costly feature improves ordering but cannot be computed for all candidates.

**Example:** *"how to combine keyword match and meaning based search"*: the relevant documents use lexical *and* semantic vocabulary. Compare which relevant document reaches the top 5 and what the second phase changes.

In [17]:
show("sparse_dense_keyword_rerank", "s14")

Strategy : sparse_dense_keyword_rerank  (rank profile: sparse_dense_keyword_rerank)
Query    : 'how to combine keyword match and meaning based search'   [designed to favour: hybrid]
Why      : Both relevant documents mix lexical and semantic vocabulary; combined signals rank them first.
YQL      : select * from doc where userQuery() or ([{"targetHits": 20}]wand(sparse_tokens, {"3642": 84, "47": 45, "70163": 195, "166117": 211, "14858": 230, "136": 14, "91084": 277, "35509": 183, "33938": 240})) or ([{"targetHits": 20}]nearestNeighbor(embedding, q))
Sparse query tokens (token, weight): [('meaning', 0.277), ('search', 0.24), ('match', 0.23), ('keyword', 0.211), ('combine', 0.195), ('based', 0.183), ('how', 0.084), ('to', 0.045), ('and', 0.014)]


,rank,id,relevant,relevance,title,match-features
0,1,sp-020,yes (2.0),1.694,Why lexical and semantic signals complement each other,"{'bm25_body': 10.11, 'bm25_title': 0.053, 'dense_score': 0.576, 'sparse_scor..."
1,2,sp-028,,1.022,Choosing a text analyzer,"{'bm25_body': 5.728, 'bm25_title': 0.0, 'dense_score': 0.433, 'sparse_score'..."
2,3,sp-017,,1.005,Designing a search results page,"{'bm25_body': 2.56, 'bm25_title': 0.816, 'dense_score': 0.509, 'sparse_score..."
3,4,sp-010,,0.901,Nearest neighbor search with HNSW graphs,"{'bm25_body': 2.357, 'bm25_title': 0.759, 'dense_score': 0.444, 'sparse_scor..."
4,5,sp-006,,0.770,Re-ranking candidates in the second phase,"{'bm25_body': 3.949, 'bm25_title': 0.0, 'dense_score': 0.374, 'sparse_score'..."


# Strategy comparison on all examples

The benchmark runs every strategy on every query and groups the nDCG@10 by the strategy the query was designed to favour. The small corpus and the strong BGE-M3 dense model mean that differences are modest; read the table for tendencies, not for statistical significance.

In [18]:
compare_ids = ["bm25_title_body", "dense_embedding", "hybrid"] + [s.id for s in STRATEGIES if s.requires_sparse]
runner = BenchmarkRunner(app, embedder=embedder)
results = runner.benchmark_strategies([strategies[i] for i in compare_ids], dataset)

favours = {q["text"]: q["favours"] for q in dataset["queries"]}
table = {}
for sid, rows in results["query_details"].items():
    by_group = {}
    for row in rows:
        by_group.setdefault(favours[row["query_text"]], []).append(row["ndcg@10"])
    table[sid] = {g: sum(v) / len(v) for g, v in by_group.items()}
    table[sid]["ALL"] = sum(r["ndcg@10"] for r in rows) / len(rows)
pd.DataFrame(table).T.round(3).sort_values("ALL", ascending=False)

No hits returned for query 'bgem3 lexicalweights' (strategy=bm25_title_body).


No hits returned for query 'hnswgraph' (strategy=bm25_title_body).


,keyword,sparse,dense,hybrid,ALL
sparse_dense,0.986,0.994,1.000,0.654,0.957
dense_embedding,0.949,0.984,1.000,0.746,0.946
sparse_dense_keyword,0.986,0.989,0.812,0.763,0.930
sparse_dense_keyword_rerank,0.986,0.989,0.812,0.763,0.930
dense_retrieve_sparse_rank,0.949,0.989,0.833,0.742,0.913
sparse_keyword,0.986,0.985,0.715,0.763,0.910
sparse_retrieve_dense_rank,0.986,0.994,0.750,0.654,0.907
sparse_dotproduct,0.949,0.989,0.750,0.721,0.895
sparse_wand,0.949,0.989,0.750,0.721,0.895
hybrid,0.986,0.985,0.623,0.728,0.888


In [19]:
pd.DataFrame(results["summary"]).drop(columns="description").set_index("strategy").round(3)

,avg_precision@10,avg_recall@10,avg_precision@1,avg_precision@3,avg_recall@5,avg_ndcg@10,avg_mrr,avg_map
strategy,,,,,,,,
bm25_title_body,0.315,0.758,0.80,0.425,0.742,0.754,0.812,0.702
dense_embedding,0.120,0.942,0.95,0.367,0.917,0.946,0.975,0.888
hybrid,0.115,0.883,0.90,0.350,0.867,0.888,0.925,0.840
sparse_wand,0.205,0.883,0.90,0.400,0.867,0.895,0.925,0.840
sparse_wand_pruned,0.204,0.867,0.90,0.417,0.867,0.885,0.925,0.836
sparse_dotproduct,0.205,0.883,0.90,0.400,0.867,0.895,0.925,0.840
sparse_field_weighted,0.205,0.883,0.80,0.383,0.867,0.857,0.860,0.795
sparse_retrieve_dense_rank,0.200,0.867,0.95,0.400,0.867,0.907,0.950,0.862
dense_retrieve_sparse_rank,0.120,0.942,0.90,0.350,0.850,0.913,0.932,0.845


## How to read the results

* **keyword** queries (identifiers, versions, rare model names): lexical and sparse signals are reliable; dense search alone can rank near-duplicate documents wrongly (`s18`). Most strategies tie here because the corpus is small.
* **sparse** queries (run-together spellings): whole-word BM25 finds nothing (`s21`, `s22`); sparse sub-word tokens and dense vectors both recover the document.
* **dense** queries (paraphrase, other language): without shared vocabulary the sparse and keyword signals have little to match; strategies with an ANN branch are needed (`s10` is solved only by strategies that retrieve with `nearestNeighbor`).
* **hybrid** queries: combining identifier evidence with semantics (`s13`) benefits from keyword/sparse signals in the mix.

BGE-M3's dense vectors are strong, so on this small corpus the dense-only strategy is hard to beat overall; the sparse strategies show their value on identifiers and spelling variants, and in combination.

Practical recommendation: use `sparse_dense_keyword` (or its rerank variant) as the default when you do not know the query mix, and prune sparse queries when latency matters. Tune the 0.3 / 0.1 mixing factors on your own relevance judgments.